# 单模态基线实验结果

**负责人**：孙钰淼（算法负责人）
**周次**：W14
**数据**：Florida 数据集（预处理后）


## 1. 实验设置

- **数据划分**：80% 训练 / 10% 验证 / 10% 测试（随机划分，seed=42）
- **结构化特征**：882 维（排除 `lastSoldPrice` 目标列和 `listPrice` 泄露变量）
- **文本特征**：
  - TF-IDF：128 维（TruncatedSVD 降维）
  - BERT 嵌入：768 维（bert-base-uncased，CLS pooling）


## 2. 基线模型介绍

本阶段仅涉及**单模态**模型，即每个模型仅使用一种类型的特征进行训练和预测。

### 2.1 结构化基线模型

| 模型 | 说明 | 定位 |
|------|------|------|
| **Linear Regression** | sklearn LinearRegression | 性能下界，任何复杂模型应优于此 |
| **Random Forest** | sklearn RandomForestRegressor（200棵树，max_depth=20） | 强基线，非线性建模 |
| **XGBoost** | XGBRegressor（500棵树，max_depth=8，lr=0.05） | 结构化数据 SOTA 基线 |

### 2.2 文本基线模型

| 模型 | 说明 | 定位 |
|------|------|------|
| **TF-IDF + Ridge** | TF-IDF(5000) → SVD(128) → Ridge(alpha=1.0) | 经典文本回归基线 |
| **BERT + MLP** | BERT CLS(768) → MLP([256,128], dropout=0.3) | 深度文本基线 |

> 注：融合模型（早期/中期/晚期）属于 W15 任务，不在此 notebook 范围内。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read experiment log, filter only baseline models
df = pd.read_csv('../results/experiment_log.csv')
baseline_names = ['LinearBaseline', 'RandomForestBaseline', 'XGBoostBaseline',
                  'TFIDFRidgeBaseline', 'BERTMLPBaseline']
df_baseline = df[df['model_name'].isin(baseline_names)]
cols = ['model_name', 'modality', 'test_rmse', 'test_mae', 'test_r2', 'test_mape']
display(df_baseline[cols].round(2))


## 3. 实验结果

### 3.1 测试集指标

| 模型 | 模态 | RMSE | MAE | R2 | MAPE(%) | 训练时间 |
|------|------|------|------|-----|---------|----------|
| LinearBaseline | structured | 138,424 | 98,184 | 0.7436 | 404.5 | 0.7s |
| XGBoostBaseline | structured | 142,935 | 99,872 | 0.7266 | 373.8 | 1.9s |
| RandomForestBaseline | structured | 164,140 | 110,771 | 0.6394 | 333.8 | 5.6s |
| TFIDFRidgeBaseline | text | 175,621 | 132,936 | 0.5872 | 501.8 | <0.1s |
| BERTMLPBaseline | text | 207,417 | 153,540 | 0.4242 | 709.9 | 41.4s |


In [ ]:
# 基线模型 R2 对比柱状图
models = ['Linear', 'RandomForest', 'XGBoost', 'TFIDF+Ridge', 'BERT+MLP']
r2_test = [0.7436, 0.6394, 0.7266, 0.5872, 0.4242]
colors = ['#3498db', '#3498db', '#3498db', '#e74c3c', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models, r2_test, color=colors, edgecolor='white', linewidth=1.2)
ax.set_ylabel('R2 Score', fontsize=12)
ax.set_title('Single-Modality Baseline Models -- Test Set R2', fontsize=14)
ax.set_ylim(0, 0.9)
for bar, score in zip(bars, r2_test):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#3498db', label='Structured'),
                   Patch(facecolor='#e74c3c', label='Text')]
ax.legend(handles=legend_elements, fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# 训练集 vs 测试集 R2 对比（检查过拟合）
import numpy as np
train_r2 = [0.8074, 0.8646, 0.8446, 0.6068, 0.7501]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, train_r2, width, label='Train R2', color='#2ecc71', edgecolor='white')
ax.bar(x + width/2, r2_test, width, label='Test R2', color='#e74c3c', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('R2 Score')
ax.set_title('Train vs Test R2 -- Overfitting Check')
ax.legend()
plt.tight_layout()
plt.show()


## 4. 结果分析

### 4.1 结构化 vs 文本

- **结构化模型普遍优于文本模型**：最佳结构化（LinearRegression, R2=0.74）远超最佳文本（TF-IDF+Ridge, R2=0.59）
- 结构化特征（面积、卧室数、建造年份等）对房价有更强的直接预测能力
- 文本描述虽含信息但单独不足以准确预测房价

### 4.2 过拟合分析

- **LinearRegression**：训练/测试差距仅 0.06，泛化最好
- **BERT+MLP 严重过拟合**：训练 R2=0.75 → 测试 R2=0.42（差距 0.33），需改进
- **TF-IDF+Ridge 最稳定**：训练/测试差距 0.02，L2 正则化有效

### 4.3 基线意义

- LinearRegression（R2=0.74）→ 融合模型性能下界
- TF-IDF+Ridge（R2=0.59）→ 文本模态性能参考
- 后续 W15 融合实验直接对比这些基线，量化多模态增量


## 5. 结论

1. **结构化特征是主力信号**（R2 0.64–0.74）
2. **文本特征单独预测弱**（R2 0.42–0.59），需在融合实验中验证增量贡献
3. **BERT+MLP 需改进**：当前过拟合严重，超参数调优空间大
4. **下一步**：进入 W15，构建早/中/晚期融合模型，对比单模态基线
